Projeto de Análises de Venda com Python

Objetivo: 
Este projeto tem como objetivo realizar o tratamento e a análise de uma base de vendas de uma empresa fictícia.
Durante o desenvolvimento serãpo aplicadas técnicas de limpeza, transformação e análise de dados utilizando Python e
Pandas, gerando indicadores que apoiam a tomada de decisão.
Tecnologias utilizadas:
- Python
- Pandas
- Numpy
- OpenPyXL
- Jupyter Notebook

In [4]:

# ===========================================================================================
# 1. IMPORTAÇÃO DAS BIBLIOTECAS
# ===========================================================================================   

import pandas as pd 
import numpy as np

# ===========================================================================================
# 2. CARREGAMENTO DOS DADOS
# ===========================================================================================   
df_vendas = pd.read_csv(
    "vendas_tech.csv", 
    encoding="latin1",
    low_memory=False
    )
df_gerentes = pd.read_excel(
    "gerentes_lojas.xlsx"
    )


print("Base de vendas carregada com sucesso!")

print(f"Linhas: {df_vendas.shape[0]}")
print(f"Colunas: {df_vendas.shape[1]}")







Base de vendas carregada com sucesso!
Linhas: 100100
Colunas: 8


In [5]:
#==========================================================================================
#3.Conhecendo os Dados
#===========================================================================================

display(df_vendas.head())


,ID_Pedido,Data,Loja,Produto,Preco_Unitario,Qtd,Cliente,Data_Base
0,1,2023-06-08,SÃ£o Paulo,Mouse Gamer,120.0,1,Cliente_4095,2025-01-01
1,2,2023-03-01,Belo Horizonte,iPhone 14,5500.0,1,Cliente_8750,NaN
2,3,2023-02-25,NaN,"Monitor 27""",1200.0,1,Cliente_14859,NaN
3,4,2024-11-19,RIO DE JANEIRO,Mouse Gamer,120.0,2,Cliente_17343,NaN
4,5,2024-01-27,Rio de Janeiro,Smartphone Samsung,2200.0,1,Cliente_23377,NaN


In [6]:
display(df_vendas.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100100 entries, 0 to 100099
Data columns (total 8 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   ID_Pedido       100100 non-null  int64  
 1   Data            100100 non-null  object 
 2   Loja            98099 non-null   object 
 3   Produto         100100 non-null  object 
 4   Preco_Unitario  100100 non-null  float64
 5   Qtd             100100 non-null  int64  
 6   Cliente         100100 non-null  object 
 7   Data_Base       1 non-null       object 
dtypes: float64(1), int64(2), object(5)
memory usage: 6.1+ MB


None

In [7]:
display(df_vendas.describe())

,ID_Pedido,Preco_Unitario,Qtd
count,100100.000000,100100.000000,100100.000000
mean,50004.810180,2000.204595,1.499101
std,28866.872543,1841.050733,1.241605
min,1.000000,40.000000,1.000000
25%,25008.750000,120.000000,1.000000
50%,50004.500000,1200.000000,1.000000
75%,75002.250000,3200.000000,1.000000
max,100000.000000,5500.000000,10.000000


In [8]:
display(df_vendas.isna().sum())

ID_Pedido              0
Data                   0
Loja                2001
Produto                0
Preco_Unitario         0
Qtd                    0
Cliente                0
Data_Base         100099
dtype: int64

4. LIMPEZA DOS DADOS: 

Nesta etapa são tratados problemas encontrados na base, como:
- Correção de caracteres;
- Remoção de valores duplicados;
- Tratamentos de valores nulos;
- Padronização do nome das lojas.


In [9]:
#==========================================================================================
#4. LIMPEZA DE DADOS
#===========================================================================================
#Corrige problemas de codificação dos nomes das lojas

df_vendas["Loja"] = (
    df_vendas["Loja"]
    .str.encode("latin1")
    .str.decode("utf-8")
)
#remove colunas que não serão utilizadas na análise
df_analise = (
    df_vendas
    .drop(columns=["Data_Base"])
    .copy()
)
#Preenche lojas sem informação com "Online" para indicar que a venda foi realizada online
df_analise["Loja"] = df_analise["Loja"].fillna("Online")

#Remove espaços antes e depois do texto
df_analise["Loja"] = df_analise["Loja"].str.strip()

#Padroniza a escrita dos nomes das lojas
df_analise["Loja"] = df_analise["Loja"].str.title()

#Converte a coluna "Data" para o formato datetime
df_analise["Data"] = pd.to_datetime(df_analise["Data"], format = "%Y-%m-%d")

#Remove pedido duplicados
df_analise = df_analise.drop_duplicates(
    subset=["ID_Pedido"]
    )


Os valores nulos da coluna Loja forma preenchidos com Online, pois de acordo com a regra de
negócio deste conjunto de dados, registros sem loja física representam vendas pelo canal online.

In [10]:
#==========================================================================================
#Validação da limpeza
#===========================================================================================
print(f"Quantidade de linhas:, {len(df_analise)}")
print(f"Quantidade de colunas:, {len(df_analise.columns)}")

print("\nValores Nulos:")
display(df_analise.isna().sum())

print("\nTipos de Colunas:")    
display(df_analise.dtypes)  



Quantidade de linhas:, 100000
Quantidade de colunas:, 7

Valores Nulos:


ID_Pedido         0
Data              0
Loja              0
Produto           0
Preco_Unitario    0
Qtd               0
Cliente           0
dtype: int64


Tipos de Colunas:


ID_Pedido                  int64
Data              datetime64[ns]
Loja                      object
Produto                   object
Preco_Unitario           float64
Qtd                        int64
Cliente                   object
dtype: object

5. Transformação de Dados: 
Nesta etapa são criadas novas variáveis que enriquecem a base de dados e facilitam as análises.
As transformações realizadas são:
- Criação da coluna faturamento;
- Classificação da forma de venda;
- Classificação da região.

In [12]:
#================================================================================ 
#5. Transformação dos dados
#================================================================================   
#Criação da coluna faturamento
df_analise['Faturamento'] = (
    df_analise['Qtd'] *
    df_analise['Preco_Unitario']
)
#Classificação da forma de venda
df_analise["Forma_de_venda"] = np.where(
    df_analise["Loja"] == "Online",
    "Online",
    "Presencial"
)
#Criação da coluna região
dic_regioes = {
    'São Paulo': 'Sudeste',
    'Belo Horizonte': 'Sudeste',
    'Online': 'Online',
    'Rio De Janeiro': 'Sudeste',
    'Salvador': 'Nordeste',
    'Recife': 'Nordeste',
    'Curitiba': 'Sul',
    'Porto Alegre': 'Sul'
}
df_analise['Regiao'] = (
    df_analise['Loja']
    .map(dic_regioes)
    .fillna('Não informada'))

#==========================================================================================
#Validação das transformações
#===========================================================================================
print("Colunas criadas com sucesso!\n")

display(
    df_analise[
        [
            "Loja",
            "Qtd",
            "Preco_Unitario",
            "Faturamento",
            "Forma_de_venda",
            "Regiao"
        ]
        ].head()
)



Colunas criadas com sucesso!



,Loja,Qtd,Preco_Unitario,Faturamento,Forma_de_venda,Regiao
0,São Paulo,1,120.0,120.0,Presencial,Sudeste
1,Belo Horizonte,1,5500.0,5500.0,Presencial,Sudeste
2,Online,1,1200.0,1200.0,Online,Online
3,Rio De Janeiro,2,120.0,240.0,Presencial,Sudeste
4,Rio De Janeiro,1,2200.0,2200.0,Presencial,Sudeste


6. KPI's do Negócio
Após a limpeza e transformação dos dados, serão calculados os principais indicadores da empresa.
Os KPI's permitem uma visão rápida do desempenho geral antes das análises detalhadas.

In [22]:
#==========================================================================================
#KPI's DO NEGÓCIO
#===========================================================================================
#KPI 1- Faturamento Total
faturamento_total = df_analise["Faturamento"].sum()

#KPI 2- Quantidade de Pedidos
total_pedidos = df_analise["ID_Pedido"].nunique()

#KPI 3- Produtos Vendidos
total_produtos= df_analise["Qtd"].sum()

#KPI 4- Ticket Médio
ticket_medio = faturamento_total / total_pedidos

#KPI 5- Número de Lojas
total_lojas = df_analise["Loja"].nunique()

In [27]:
#==========================================================================================
#Painel de KPI's
#===========================================================================================

print("=" * 45)
print("                          INDICADORES DE NEGÓCIO")
print("=" * 45)

print(f"💰 Faturamento Total : R$ {faturamento_total:,.2f}")
print(f"🛒 Total de Pedidos : {total_pedidos}")
print(f"📦 Produtos Vendidos: {total_produtos}")
print(f"🏬 Total de Lojas   : {total_lojas}")
print(f"💳 Ticket Médio     : R$ {ticket_medio:,.2f}")


print("=" * 45)

                          INDICADORES DE NEGÓCIO
💰 Faturamento Total : R$ 299,472,330.00
🛒 Total de Pedidos : 100000
📦 Produtos Vendidos: 149902
🏬 Total de Lojas   : 8
💳 Ticket Médio     : R$ 2,994.72


7. ANÁLISE E GERAÇÃO DE INSIGHTS
Nesta etapa serão realizadas análises exploratórias para identificar padrões de vendas,
desempenho das lojas, regiões e produtos.

In [28]:
#================================================================================
#7. Análise e geração de insights
#================================================================================
#Qual loja teve o maior faturamento?

analise_por_loja = (
    df_analise
    .groupby("Loja", as_index=False)
    .agg(
        Faturamento=("Faturamento", "sum")
    )
    .sort_values("Faturamento", ascending=False)
)
display(analise_por_loja)




,Loja,Faturamento
6,Salvador,42300610.0
5,Rio De Janeiro,42294720.0
4,Recife,42190390.0
7,São Paulo,42090690.0
0,Belo Horizonte,41714890.0
3,Porto Alegre,41678460.0
1,Curitiba,41121720.0
2,Online,6080850.0


### Insight
A análise do faturamento por loja permite identificar quais unidades concentram a maior receita da empresa.
Essas informações podem apoiar decisões como investimentos, campanhas comerciais e expansão das oeprações.

In [31]:
#Qual região vende mais?
analise_regiao = (
    df_analise
    .groupby("Regiao", as_index=False)
    .agg(
        Faturamento=("Faturamento","sum")
    )
    .sort_values("Faturamento", ascending=False)
    )

display(analise_regiao)



,Regiao,Faturamento
2,Sudeste,126100300.0
0,Nordeste,84491000.0
3,Sul,82800180.0
1,Online,6080850.0


In [35]:
#Canal Online x Presencial
analise_canal = (
    df_analise
    .groupby("Forma_de_venda", as_index=False)
    .agg(
        Faturamento=("Faturamento", sum),
        Pedidos=("ID_Pedido", "nunique")
    )
)
display(analise_canal)

C:\Users\Home\AppData\Local\Temp\ipykernel_18224\2665220706.py:5: FutureWarning: The provided callable <built-in function sum> is currently using SeriesGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  .agg(


,Forma_de_venda,Faturamento,Pedidos
0,Online,6080850.0,2001
1,Presencial,293391480.0,97999


In [37]:
#Produto mais vendido
top_produtos = (
    df_analise
    .groupby("Produto", as_index=False)
    .agg(
        Quantidade=("Qtd", "sum")
    )
    .sort_values("Quantidade", ascending=False)
    )
display(top_produtos.head(10))

,Produto,Quantidade
1,"Monitor 27""",19024
4,Notebook HP,18899
7,iPhone 14,18820
0,Cabo HDMI,18804
6,Teclado MecÃ¢nico,18783
2,Mouse Gamer,18771
3,Notebook Dell,18457
5,Smartphone Samsung,18344


In [38]:
#Evolução das Vendas ao longo do tempo

vendas_mensais = (
    df_analise
    .groupby(df_analise["Data"].dt.to_period("M"))
    .agg(
        Faturamento=("Faturamento", "sum")
    )
)
display(vendas_mensais)

,Faturamento
Data,
2023-01,12930290.0
2023-02,11515150.0
2023-03,12516080.0
2023-04,12528900.0
2023-05,12940470.0
2023-06,12455820.0
2023-07,12550990.0
2023-08,12989130.0
2023-09,12118180.0


In [6]:
import pandas as pd
import numpy as np

print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)

Pandas: 2.3.1
NumPy: 2.3.2


In [2]:
print("Notebook funcionando!")

Notebook funcionando!
